> ⚠️ **시작하기 전에 — 클러스터 연결**
>
> 이 실습에는 **Photon이 비활성화된 클래식 컴퓨팅 클러스터**가 필요합니다. 서버리스 컴퓨팅은 이 연습에서 사용되는 Spark 구성 옵션을 지원하지 않으며, Photon은 이 실습이 노출하도록 설계된 왜곡 및 셔플 문제를 자동으로 최적화합니다. 중 하나라도 활성 상태이면 Spark UI에서 비정상적인 것을 볼 수 없습니다.
>
> 1. 이 노트북 상단의 컴퓨팅 선택기에서 **연결**을 클릭합니다.
> 2. **`perf-lab`** 클러스터를 선택합니다(실습 설정 지침에서 생성됨 — Photon 없음 런타임, 스파크 구성에서 `spark.databricks.photon.enabled false` 설정).
> 3. 셀을 실행하기 전에 클러스터 상태가 **실행 중**으로 표시될 때까지 기다립니다.

## 연습 1: 환경 설정

성능 문제를 조사하기 전에 작동할 데이터가 필요합니다. 이 연습은 Unity Catalog 구조를 생성하고 2개의 Delta 테이블을 생성합니다:

- **`perf_lab.data.transactions`** — 500,000행(의도적으로 왜곡된 `customer_id` 열, ~70%의 행이 단일 고객에 속함)
- **`perf_lab.data.customers`** — 200행(작은 차원 테이블)

이 연습의 모든 셀을 위에서 아래로 실행한 후 계속하십시오.

### 작업 1.1: Unity Catalog 구조 생성

이 셀은 제공됩니다. 실행하여 카탈로그 및 스키마를 생성합니다.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

In [ ]:
# ✅ Run this cell — creates the catalog and schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS perf_lab MANAGED LOCATION '{storage_root}' COMMENT 'Performance troubleshooting lab'")
spark.sql("CREATE SCHEMA IF NOT EXISTS perf_lab.data COMMENT 'Transaction data for performance exercises'")
print("✅ Catalog and schema ready.")